# SCI1070 — Semaine 9 : politique déterministe, et la surestimation mesurée

| Volet | Ce que vous y faites |
| --- | --- |
| A | la boucle en neuf étapes de la section 9.3, sur `Pendulum-v1` |
| B | **la surestimation, mesurée par vous** : le nombre que la section 9.5 n'a pas donné |
| C | l'ablation des trois correctifs, et les deux bruits d'exploration |
| Démo | un robot simulé, rendu hors écran, exporté en vidéo |

**La tâche de mesure est `Pendulum-v1`.** Contrôle classique, sans moteur physique
externe, **une seule dimension d'action**, un couple dans \([-2,2]\). Le pendule
inversé de MuJoCo, `InvertedPendulum-v5`, **sature** : une ablation menée dessus ne
mesurerait rien.

> **Trois pièges, et ils coûtent chacun une soirée.**
>
> 1. **Les deux optimiseurs.** La rétropropagation de la perte de l'acteur traverse
>    le critique et y dépose des gradients. Sans remise à zéro explicite, votre
>    critique apprend à **maximiser sa propre sortie**.
> 2. **L'évaluation avec bruit.** Vous mesureriez votre réglage de bruit autant que
>    votre politique.
> 3. **L'échelle de l'action.** Multipliez par la demi-étendue **une fois, au bon
>    endroit**, ou votre agent n'aura jamais accès qu'à la moitié de son autorité.

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)
if importlib.util.find_spec("sci1070") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "git+https://github.com/FaycalZM/drl-course-teluq.git#subdirectory=code"],
                   check=True)

import collections
import copy

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sci1070.protocole import intervalle, se_recouvrent
from sci1070.courbes import tracer_bande
from sci1070.graines import fixer_graines

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRAINES = [0, 1, 2]

env_mesure = gym.make("Pendulum-v1")
DEMI_ETENDUE = float(env_mesure.action_space.high[0])
print("Pendulum-v1 | action dans "
      f"[{env_mesure.action_space.low[0]}, {env_mesure.action_space.high[0]}]"
      f" | demi-étendue = {DEMI_ETENDUE}")
env_mesure.close()

---
# Volet A : la boucle déterministe

**Étape 1 — les deux réseaux.** Gardez les tailles **400 et 300**.

In [ ]:
class Acteur(nn.Module):
    """De l'observation vers UNE action, déterministe. Tanh en sortie."""

    def __init__(self, n_obs, n_actions, demi_etendue):
        super().__init__()
        self.demi_etendue = float(demi_etendue)
        ...        # À COMPLÉTER : 400, 300, puis tanh. Piège nº3 : où multipliez-vous par la demi-étendue ? UNE seule fois.


class Critique(nn.Module):
    """Deux entrées : l'observation ET l'action. Une sortie scalaire."""

    def __init__(self, n_obs, n_actions):
        super().__init__()
        ...        # À COMPLÉTER : l'action entre dans le réseau, elle n'est pas un indice

**Étape 2 — le tampon de rejeu.** La **seule** différence avec la semaine 5 : le
champ `action` est un vecteur de flottants, donc de forme `(capacite, n_actions)`.

In [ ]:
class TamponContinu:
    def __init__(self, capacite, graine=0):
        ...        # À COMPLÉTER : le tampon de la semaine 5, avec actions continues

**Étape 3 — le bruit d'exploration.** Gaussien **non corrélé**, plus une phase de
**démarrage uniforme**.

In [ ]:
def agir_avec_bruit(acteur, obs, env, sigma_fraction=0.1):
    """Rend (action_envoyee, action_proposee). L'écart des deux est l'amplitude effective."""
    ...        # À COMPLÉTER : le bruit, et l'écrêtage


def agir_sans_bruit(acteur, obs, env, device):
    """L'évaluation se fait TOUJOURS par cette fonction. Piège nº2."""
    with torch.no_grad():
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        return acteur(obs_t).cpu().numpy()[0]

**Étape 4 — les deux pas d'optimisation**, chacun avec son optimiseur.

In [ ]:
def pas_optimisation(lot, acteur, critique, acteur_cible, critique_cible,
                     critique2, critique2_cible, opt_acteur, opt_critique,
                     gamma=0.99, tau=0.005, double_q_ecrete=False,
                     delai_politique=False, lissage_cible=False, iteration=0,
                     demi_etendue=2.0):
    ...        # À COMPLÉTER : les deux pertes, les deux optimiseurs, la synchro douce. Les trois derniers indicateurs sont les interrupteurs du volet C : écrivez-les MAINTENANT.

### Les quatre vérifications arithmétiques

In [ ]:
# 1. L'action envoyée doit TOUJOURS être dans [-2, 2].
env = gym.make("Pendulum-v1")
basse, haute = env.action_space.low, env.action_space.high
action_test = np.array([1.7], dtype=np.float32)
assert np.all(action_test >= basse) and np.all(action_test <= haute)

# 4. La cible et la prédiction doivent avoir EXACTEMENT la même forme.
cible = torch.zeros(100, 1)
prediction = torch.zeros(100)
print("forme de (cible - prediction) :", (cible - prediction).shape, " <- le piège")
assert (cible - prediction).shape == (100, 100)
print("d'où la règle : garder (N, 1) des deux côtés, une fois pour toutes")
env.close()

In [ ]:
def entrainer(graine, max_pas=40_000, double_q_ecrete=False, delai_politique=False,
              lissage_cible=False, sigma_fraction=0.1, demarrage=1_000,
              evaluer_tous=2_000, etats_temoins=None, gamma=0.99):
    """Rend (courbe_evaluation, journal). UN programme, trois interrupteurs."""
    ...        # À COMPLÉTER : la boucle en neuf étapes de la section 9.3


courbe_pilote, journal_pilote = entrainer(graine=0)
print("cible annoncée à partir de l'exécution pilote :", round(float(courbe_pilote.max()), 1))
print("amplitude de bruit effective :",
      round(float(np.mean(journal_pilote["amplitude_effective"])), 4))

In [ ]:
courbes = [entrainer(graine=g)[0] for g in GRAINES]

fig, ax = plt.subplots(figsize=(8, 4.5))
tracer_bande(ax, np.arange(len(courbes[0])), courbes, etiquette="base")
ax.set_xlabel("évaluation"); ax.set_ylabel("retour, sans bruit, dix épisodes")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Attentes du volet A.** La courbe d'évaluation sur au moins trois graines avec
intervalle, **votre amplitude de bruit effective**, et votre cible annoncée.

> **Sur la vérification 2, le signe.** Les récompenses de `Pendulum-v1` sont
> **négatives** : une valeur prédite positive est un bogue de signe, pas un
> apprentissage optimiste. Et au tout début, la valeur prédite doit être **proche de
> zéro**, puisque les poids sont initialisés petits.

---
# Volet B : la surestimation, mesurée par vous

**Étape 1.** Après la phase de démarrage, tirez un ensemble d'états et **figez-le**.

In [ ]:
def figer_etats_temoins(graine=0, n=300):
    """Tirés une fois, conservés pour toute l'exécution. Jamais retirés."""
    ...        # À COMPLÉTER : quelques centaines d'états, tirés d'une politique uniforme


etats_temoins = figer_etats_temoins()
print("états témoins :", tuple(etats_temoins.shape))

**Étape 2.** Valeur moyenne estimée par le critique, et retour actualisé
**réellement collecté** en suivant la politique **sans bruit**.

> **Une mise en garde sur le coût.** Ne mesurez pas sur mille épisodes : mesurez sur
> une dizaine, et **dites dans votre rapport que votre moyenne porte sur dix
> épisodes**. Une mesure bruitée et annoncée comme telle vaut infiniment mieux qu'une
> mesure propre et non reproductible.

In [ ]:
def retour_reel(acteur, gamma=0.99, episodes=10, graine=0):
    """Joue SANS bruit et actualise. C'est une MESURE, pas un entraînement."""
    ...        # À COMPLÉTER : jouer sans bruit, et actualiser

**Étape 4.** Refaites la mesure avec le **double Q écrêté** activé, **sur les mêmes
graines**.

In [ ]:
...        # À COMPLÉTER : les deux courbes, estimée et réelle, dans les deux configurations

**Répondez à trois questions.** La surestimation apparaît-elle sur votre tâche ? À
quel moment ? Et son signe est-il **stable**, ou votre critique sous-estime-t-il
parfois ?

---
# Volet C : l'ablation des trois correctifs

**Un seul programme, trois interrupteurs.**

In [ ]:
CONFIGURATIONS = {
    "Base":                dict(double_q_ecrete=False, delai_politique=False, lissage_cible=False),
    "Base + correctif 1":  dict(double_q_ecrete=True,  delai_politique=False, lissage_cible=False),
    "Base + correctif 2":  dict(double_q_ecrete=False, delai_politique=True,  lissage_cible=False),
    "Base + correctif 3":  dict(double_q_ecrete=False, delai_politique=False, lissage_cible=True),
    "Les trois":           dict(double_q_ecrete=True,  delai_politique=True,  lissage_cible=True),
}

resultats = {}
for nom, options in CONFIGURATIONS.items():
    scores = ...        # À COMPLÉTER : un score final par graine, évaluation SANS bruit
    resultats[nom] = scores
    moyenne, bas, haut = intervalle(scores)
    print(f"{nom:22s} {moyenne:8.1f}   [{bas:.1f}, {haut:.1f}]   n={len(scores)}")

In [ ]:
base = resultats["Base"]
for nom in list(CONFIGURATIONS)[1:]:
    recouvre = se_recouvrent(base, resultats[nom])
    verdict = "je ne mesure pas de différence" if recouvre else "séparé de la base"
    print(f"{nom:22s} {verdict}")

**Étape 4, la lecture.** Pour chacun des trois correctifs : aide-t-il, ne change-t-il
rien, ou **nuit-il** ? Et surtout : votre tâche à **une** dimension d'action
souffre-t-elle du défaut de la section 9.5 ? **Si votre volet B montre peu de
surestimation, vous devez vous attendre à ce que le correctif 1 n'apporte rien, et
l'observer serait une confirmation, pas un échec.**

Dites aussi ce que trois graines vous autorisent à conclure **de moins** que dix :
avec trois tirages, l'intervalle par rééchantillonnage est large et ne sépare que
des écarts grossiers ; dix resserreraient l'intervalle d'un facteur proche de
\(\sqrt{10/3}\).

### Étape 5 — la comparaison des deux bruits

In [ ]:
# La solution courte, pour verifier votre implementation et comparer les bruits.
# A executer APRES avoir ecrit la votre, jamais avant.
import numpy as np
from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

n_actions = 1
sigma = 0.1 * np.ones(n_actions)

bruit_simple = NormalActionNoise(mean=np.zeros(n_actions), sigma=sigma)
bruit_correle = OrnsteinUhlenbeckActionNoise(mean=np.zeros(n_actions), sigma=sigma)

for nom, bruit in [("non correle", bruit_simple), ("correle", bruit_correle)]:
    modele = TD3("MlpPolicy", "Pendulum-v1", action_noise=bruit, seed=0, verbose=0)
    modele.learn(total_timesteps=30_000)
    # Puis evaluer SANS bruit, sur plusieurs graines, et comparer.

**Si vous ne trouvez aucune différence, vous venez de retrouver par vous-même ce que
les trois constats de la section 9.4 annonçaient**, et c'est le meilleur résultat
possible pour cet exercice.

---
# Démonstration : un robot, en vidéo, sans écran

In [ ]:
if importlib.util.find_spec("mujoco") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "gymnasium[mujoco]"], check=True)

import gymnasium as gym

# Verifiez l'identifiant exact dans VOTRE installation : les suffixes de version
# des environnements MuJoCo changent d'une version de Gymnasium a l'autre, et
# gymnasium.pprint_registry() en donne la liste.
ID_ROBOT = "Hopper-v5"

# Rendu hors ecran : les images sont renvoyees en memoire, jamais affichees.
env = gym.make(ID_ROBOT, render_mode="rgb_array")
env = gym.wrappers.RecordVideo(env, video_folder="videos", name_prefix="demo")

obs, _ = env.reset(seed=0)
acteur_robot = Acteur(env.observation_space.shape[0],
                      env.action_space.shape[0],
                      float(env.action_space.high[0])).to(device)
termine = tronque = False
while not (termine or tronque):
    action = agir_sans_bruit(acteur_robot, obs, env, device)
    obs, _, termine, tronque, _ = env.step(action)
env.close()        # INDISPENSABLE : c'est close() qui finalise le fichier video.

**Attendu.** Une vidéo, et **deux phrases**. Le robot qui tombe vers l'avant au bout
de onze secondes est un comportement **appris** et non un échec : c'est une démarche,
instable, et la décrire est un exercice d'observation que ce cours n'a pas encore
demandé.

> **Mon message.** Faites les volets **dans l'ordre**. Vous n'avez, dans cette
> famille de méthodes, **aucune** perte dont la valeur vous dise si les choses se
> passent bien. **Le volet C ne devient interprétable qu'après le volet B.**